In [5]:
import pandas as pd

df = pd.read_csv("../data/processed/monthly_sales.csv")
df["date"] = pd.to_datetime(df["date"])

ts = df.set_index("date")["sales"]

# Train-test split
train_size = int(len(ts) * 0.8)
train = ts[:train_size]
test = ts[train_size:]

In [6]:
prophet_df = df.rename(columns={"date": "ds", "sales": "y"})

In [2]:
import cmdstanpy
cmdstanpy.install_cmdstan()

CmdStan install directory: C:\Users\SRIDHAR\.cmdstan
Installing CmdStan version: 2.38.0
Download successful, file: C:\Users\SRIDHAR\AppData\Local\Temp\tmptrdvtap1
Extracting distribution


21:45:40 - cmdstanpy - WARNING - CmdStan installation failed.
Command "make build" failed
Command: ['mingw32-make', 'build', '-j1']
failed with error [WinError 2] The system cannot find the file specified



Unpacked download as cmdstan-2.38.0
Building version cmdstan-2.38.0, may take several minutes, depending on your system.


False

In [7]:
train_prophet = prophet_df.iloc[:train_size]

from prophet import Prophet

model = Prophet()
model.fit(train_prophet)

21:49:59 - cmdstanpy - INFO - Chain [1] start processing
21:49:59 - cmdstanpy - INFO - Chain [1] done processing


In [8]:
future = model.make_future_dataframe(periods=len(test), freq="M")
forecast = model.predict(future)

In [9]:
future = model.make_future_dataframe(periods=len(test), freq="M")
forecast = model.predict(future)

pred_prophet = forecast["yhat"][-len(test):]

In [12]:
prophet_pred_df = pd.DataFrame({
    "date": test.index,
    "actual": test.values,
    "prophet_pred": pred_prophet
})

prophet_pred_df.to_csv("../outputs/metrics/prophet_predictions.csv", index=False)

In [11]:
import joblib
joblib.dump(model, "../models/prophet.pkl")

['../models/prophet.pkl']